# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A7mad7-7/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

### Feature Vector Assembly Plan
This section constructs a robust feature vector from raw search warehouse logs. It applies log-transformations for heavily skewed demand distribution, creates categorical encoding for page taxonomy, and imputes missing numeric records to maintain data integrity.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load dataset slice
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Filter for valid active rows
mask = (df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)
df_valid = df[mask].copy()

# 1. Engineer Features
X = pd.DataFrame()
X['content_id'] = df_valid['content_id']

# Numeric log-transformed demand features
X['f_log_impressions_90d'] = np.log1p(df_valid['impressions_90d'].fillna(0))
X['f_log_search_volume'] = np.log1p(df_valid['search_volume'].fillna(0))

# Recency & Engagement Features
X['f_days_since_update'] = df_valid['days_since_last_update'].fillna(df_valid['days_since_last_update'].median())
X['f_click_through_rate'] = (df_valid['clicks_90d'].fillna(0) / (df_valid['impressions_90d'] + 1e-5)).clip(0, 1)

# Categorical Handling (One-hot encoding)
X['f_is_blog'] = (df_valid['content_type'] == 'blog').astype(int)

print(f"Feature Vector Built Successfully. Shape: {X.shape}")
print(X.head())

Feature Vector Built Successfully. Shape: (30000, 6)
             content_id  f_log_impressions_90d  f_log_search_volume  \
0  content_304f48230142               8.243808             2.397895   
1  content_a1fb4e703a9e               9.636980             4.510860   
2  content_9aa793d4d895               9.440023             0.000000   
3  content_331d6c4de07b               9.371779             2.397895   
4  content_d99b7a2d90ca               9.859588             0.000000   

   f_days_since_update  f_click_through_rate  f_is_blog  
0                   20              0.007626          0  
1                   25              0.000457          0  
2                   20              0.000874          0  
3                   22              0.004936          0  
4                   14              0.001254          0  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Metadata Audit Table

| Feature Name | Meaning | Missing Value Handling | Available When? |
| :--- | :--- | :--- | :--- |
| `f_log_impressions_90d` | Log-scaled total 90-day search impressions | Imputed with 0 via `np.log1p` | Before decision (Historical Search Console logs) |
| `f_log_search_volume` | Log-scaled target keyword search volume | Imputed with 0 prior to transformation | Before decision (Pre-computed search index) |
| `f_days_since_update` | Elapsed days since last CMS update | Imputed with median age | Before decision (Internal CMS metadata timestamp) |
| `f_click_through_rate` | Historical CTR (`clicks_90d` / `impressions_90d`) | Zero-filled division protection | Before decision (Historical Search Console logs) |
| `f_is_blog` | One-hot flag for blog taxonomy | Defaulted to 0 (non-blog) | Before decision (Static URL structure/CMS metadata) |

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify completeness and absence of missing values across features
missing_summary = X.isnull().sum()
print("Missing values per engineered feature:")
print(missing_summary)

Missing values per engineered feature:
content_id               0
f_log_impressions_90d    0
f_log_search_volume      0
f_days_since_update      0
f_click_through_rate     0
f_is_blog                0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### Leakage Attack Test
We test our feature vector against two classic leakage patterns:
1. **Target-Derived Feature**: Including the downstream decay status `trend_direction == 'down'`.
2. **Future Window Leakage**: Using metrics derived from future evaluation intervals (e.g., June 2026 logs).

Below, we simulate the leak, observe artificial metric inflation, and strip the leaked fields to restore a clean decision boundary.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Compute ground-truth proxy target
y_true = np.where(df_valid['trend_direction'] == 'down', 1, 0)

# Simulate Leakage Test
X_test = X.copy()
X_test['LEAK_future_trend'] = y_true  # Direct target leak

# Measure correlation of features with target
correlations = X_test.drop(columns=['content_id']).apply(lambda col: np.corrcoef(col, y_true)[0, 1])

print("Feature Correlations with Target (Notice LEAK_future_trend):")
print(correlations)

# Clean leakage column
X_clean = X_test.drop(columns=['LEAK_future_trend'])
print("\n[LEAKAGE REMOVED]: Clean feature frame restored without target-derived columns.")

Feature Correlations with Target (Notice LEAK_future_trend):
f_log_impressions_90d    0.177473
f_log_search_volume     -0.057901
f_days_since_update      0.081383
f_click_through_rate    -0.061909
f_is_blog                     NaN
LEAK_future_trend        1.000000
dtype: float64

[LEAKAGE REMOVED]: Clean feature frame restored without target-derived columns.


/home/ahmad/Documents/AI_Projects/FlyRank_ML_Internshipe/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ahmad/Documents/AI_Projects/FlyRank_ML_Internshipe/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Fields & Justification

* `client_id`: Excluded to prevent client-specific overfitting and preserve privacy.
* `url_path_raw`: Excluded raw URL string to avoid PII/privacy exposure and high-cardinality noise.
* `future_clicks_june`: Excluded as it belongs to the sealed evaluation window (violates temporal boundary).
* `post_refresh_position`: Excluded because post-update ranking metrics only exist AFTER the intervention has already happened.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Document excluded fields check
excluded_fields = ['client_id', 'url_path_raw', 'future_clicks_june', 'post_refresh_position']
present_in_vector = [field for field in excluded_fields if field in X_clean.columns]

print(f"Excluded fields present in feature vector: {len(present_in_vector)} (Expected: 0)")

Excluded fields present in feature vector: 0 (Expected: 0)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support[cite: 1]
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.